# Lunar Mascon Orbit Simulator - Analysis & Visualization

Front-end for the C++ symplectic propagator (`lunarmc`).

**Pipeline:** export GRGM1200B gravity + LOLA topography &rarr; build C++ &rarr; run Monte-Carlo over start (lat, lon) and altitude [25, 250] km &rarr; plot.

Physics improvements vs the old `moon_orbit_v0.py`:
- full spherical-harmonic sum (no `m in {0,1,2,l}` truncation, no radial-only gradient),
- **symplectic** velocity-Verlet / Yoshida-4 &rarr; total specific energy conserved (bounded, no secular drift),
- topography-aware impact: satellite falls when `r < R_surface(lat, lon)`.

**Revised 2026-10-13:** GRGM1200B non-zonal normalization and time-dependent integrators corrected. Previous Monte Carlo figures are superseded; local impact angles use the interpolated terrain normal, not the radial direction. No 59 m DEM is available in this project.


In [ ]:
import os, subprocess, struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = os.path.abspath('..')
os.chdir(ROOT)
print('project root:', ROOT)

## 1. Export gravity + topography (run once)
Needs `pyshtools`; downloads GRGM1200B & MoonTopo2600p the first time.
For a quick smoke test without pyshtools, run `tools/make_synth_data.py` instead (J2-only).

In [ ]:
# Real datasets (slow first run). Lower --lmax for speed.
subprocess.run(['python', 'tools/export_data.py', '--lmax', '300', '--ltopo', '180'], check=True)
# Quick alternative:
# subprocess.run(['python', 'tools/make_synth_data.py'], check=True)

## 2. Build the C++ propagator

In [ ]:
subprocess.run(['cmake', '-B', 'build', '-S', '.', '-DCMAKE_BUILD_TYPE=Release'], check=True)
subprocess.run(['cmake', '--build', 'build', '-j', '4'], check=True)

## 3. Run the Monte-Carlo
Tune `--n` (samples), `--lmax` (fidelity vs speed), `--days`, `--integrator`.

In [ ]:
os.makedirs('results', exist_ok=True)
subprocess.run(['./build/lunarmc',
                '--n', '500', '--lmax', '120', '--days', '30',
                '--alt-min', '25', '--alt-max', '250',
                '--integrator', 'yoshida4', '--traj-count', '12'], check=True)

In [ ]:
summary = pd.read_csv('results/summary.csv')
traj    = pd.read_csv('results/trajectories.csv')
print(f"impacts: {summary.impacted.sum()}/{len(summary)} "
      f"({100*summary.impacted.mean():.1f}%)")
print(f"worst invariant drift: {summary.invariant_drift_rel.max():.2e}")
summary.head()

### Load topography grid (for background maps)

In [ ]:
def load_topo(path='data/topo.bin'):
    with open(path, 'rb') as f:
        nlat, nlon = struct.unpack('<ii', f.read(8))
        d = np.fromfile(f, dtype='<f8', count=nlat*nlon).reshape(nlat, nlon)
    return d
topo = load_topo()
topo_km = topo/1e3
extent = [0, 360, -90, 90]  # lon, lat

## 4. Jacobi-invariant numerical diagnostic (not proof of symplecticity)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for i, g in traj.groupby('id'):
    ax[0].plot(g.t_hr, (g.invariant - g.invariant.iloc[0]) / abs(g.invariant.iloc[0]))
ax[0].set(xlabel='time [hr]', ylabel='(I - I0)/|I0|',
          title='Relative invariant (energy / Jacobi) drift per orbit')
ax[1].hist(np.log10(summary.invariant_drift_rel.clip(1e-16)), bins=30)
ax[1].set(xlabel='log10 max |dI/I|', ylabel='count',
          title='Energy-drift distribution')
plt.tight_layout(); plt.show()

## 5. Impact map over lunar topography

In [ ]:
imp = summary[summary.impacted == 1].copy()
imp['lon360'] = imp.impact_lon % 360
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(topo_km, extent=extent, origin='upper', cmap='gray', aspect='auto')
sc = ax.scatter(imp.lon360, imp.impact_lat, c=imp.t_end_hr/24,
                cmap='plasma', s=28, edgecolor='k', linewidth=0.3)
plt.colorbar(im, ax=ax, label='surface radius [km]', fraction=0.025)
plt.colorbar(sc, ax=ax, label='time to impact [days]', fraction=0.025)
ax.set(xlabel='longitude [deg]', ylabel='latitude [deg]',
       title='Impact sites (colour = time to impact)')
plt.tight_layout(); plt.show()

## 6. Ground tracks of sampled trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(topo_km, extent=extent, origin='upper', cmap='gray', aspect='auto', alpha=0.7)
for i, g in traj.groupby('id'):
    ax.scatter(g.lon % 360, g.lat, s=2, label=f'sat {i}')
ax.set(xlabel='longitude [deg]', ylabel='latitude [deg]', title='Ground tracks')
plt.tight_layout(); plt.show()

## 7. Altitude & eccentricity evolution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for i, g in traj.groupby('id'):
    ax[0].plot(g.t_hr/24, g.alt_km)
    ax[1].plot(g.t_hr/24, g.ecc)
ax[0].axhline(0, color='k', lw=0.8, ls='--')
ax[0].set(xlabel='time [days]', ylabel='altitude [km]', title='Altitude')
ax[1].set(xlabel='time [days]', ylabel='eccentricity', title='Eccentricity growth (mascons)')
plt.tight_layout(); plt.show()

## 8. Survival statistics vs starting altitude / inclination

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(summary.alt0, summary.t_end_hr/24, c=summary.impacted, cmap='coolwarm', s=12)
ax[0].set(xlabel='start altitude [km]', ylabel='time to impact / survival [days]',
          title='Lifetime vs start altitude')
ax[1].scatter(summary.inc0, summary.t_end_hr/24, c=summary.impacted, cmap='coolwarm', s=12)
ax[1].set(xlabel='inclination [deg]', ylabel='lifetime [days]',
          title='Lifetime vs inclination')
ax[2].hist(summary[summary.impacted==1].t_end_hr/24, bins=30)
ax[2].set(xlabel='time to impact [days]', ylabel='count', title='Impact-time distribution')
plt.tight_layout(); plt.show()